# Experimentos reproducibles · Grupo 4
Comparación B1 a 640, M1 a 1280 y M2 con entrenamiento multiescala. Seleccione GPU y configure el almacenamiento de resultados.

In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO_URL = "https://github.com/therockbeats/final-mia203.git"
REPO_REF = "main" # sustituir por commit o v1.0-final en la entrega
REPO_DIR = Path("/content/ExamenFinal")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "scripts/install_colab.py"], check=True)
print("Reinicie el entorno de Colab si cambió PyTorch; luego continúe desde la siguiente celda.")


## Datos y almacenamiento persistente
Ajuste las rutas a su Drive. La configuración local se regenera después de reiniciar Colab.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json
REPO_DIR = Path("/content/ExamenFinal")
os.chdir(REPO_DIR)
drive.mount("/content/drive")
# Puede usar la ruta de un acceso directo al Drive compartido.
WORK = Path("/content/drive/MyDrive/ExamenFinal_artefactos")
DATASET_ZIP = Path("/content/drive/MyDrive/Examen Final/data/processed.zip")
LOCAL_DATA = Path("/content/ExamenFinal_data/processed")
settings = {"dataset": str(LOCAL_DATA), "runs": str(WORK/"runs"),
    "yolo": "/content/yolov5_final", "initial_weights": str(WORK/"weights/yolov5s.pt"),
    "audit": str(WORK/"data/audit.json")}
Path("configs/paths.local.json").write_text(json.dumps(settings, indent=2))
def pipeline(*args):
    subprocess.run([sys.executable, "-m", "src.pipeline", *map(str,args)], check=True)
def reproduce(phase, split="val", seeds=(0,)):
    subprocess.run([sys.executable, "scripts/reproduce.py", phase, "--split", split,
        "--seeds", *map(str,seeds)], check=True)


In [ ]:
if not LOCAL_DATA.exists():
    pipeline("extract", "--zip", DATASET_ZIP)
pipeline("audit")
pipeline("prepare")
pipeline("doctor")


## Incorporar los modelos del grupo
REUSE_GROUP_MODELS permite elegir entre reutilizar B1/M1/M2 o entrenar réplicas propias.


In [ ]:
SHARED_SOURCE = Path("/content/drive/MyDrive/Examen Final")
REUSE_GROUP_MODELS = True
if REUSE_GROUP_MODELS:
    for experiment, command in [("B1", "import-b1"), ("M1", "import-m1"), ("M2", "import-m2")]:
        if not (WORK/"runs"/experiment/"seed0/run").exists():
            pipeline(command, "--source", SHARED_SOURCE)


## Comprobación corta y entrenamiento
B1 y M1 utilizan lote 16. Compruebe la memoria disponible con la prueba corta antes de iniciar el entrenamiento.

In [ ]:
pipeline("train", "B1", "--smoke")
pipeline("train", "M1", "--smoke")
pipeline("train", "M2", "--smoke")
reproduce("train") # reanuda una ejecución parcial; reutiliza una completa


## Validación y decisiones
Compare las métricas por clase, los errores por tamaño y el costo de los tres experimentos.

In [ ]:
reproduce("validation")


## Congelar el protocolo
Guarde la configuración seleccionada con validación. El archivo frozen_protocol.json permite utilizarla en la evaluación final.

In [ ]:
RATIONALE = "M2 seleccionado con validación entre B1/M1/M2"
frozen = WORK/"frozen_protocol.json"
if frozen.exists():
    import shutil
    shutil.copy2(frozen, "configs/frozen_protocol.json")
else:
    pipeline("freeze", "--seeds", 0, "--experiments", "M2", "--rationale", RATIONALE)
    import shutil
    shutil.copy2("configs/frozen_protocol.json", frozen)


## Evaluación final
Evalúe las configuraciones seleccionadas sobre la partición de prueba y exporte las tablas y figuras.

In [ ]:
reproduce("test")
import shutil
published = WORK/"publication_results"
shutil.copytree("results", published, dirs_exist_ok=True)
print("Tablas, figuras y errores:", published)
